# Potential self-driving routes: construction on each route, 2017–2021

For each candidate route out of San Francisco: how much road construction sat on it, how much of that was
high-severity, and how much happened in weather that makes self-driving hard.
Every number comes from the dbt model `route_construction_summary` (one row per route), which is built on
`construction_route_overlap` (each construction matched to the routes its 10 m-wide area touches).

- **Construction hours**: sum of each construction's duration (end − start). The full duration counts, not
  prorated by how much of the construction lies on the route.
- **High severity hours**: construction hours where `severity` is 3 or 4. Severity is the construction's
  impact on traffic, from 1 (short delay) to 4 (long delay).
- **High/Severe weather impact hours**: construction hours where `self_driving_impact_level` is High or Severe.
- A construction on a shared stretch (e.g. the Bay Bridge, used by Berkeley and San Leandro) counts on every
  route it touches.
- **Weak matches are dropped**: a construction counts on a route only if it lies fully on the route or at
  least 50 m of it overlaps the route. Constructions that only cross the route, or only start or end at it,
  mostly lie on another road.

## 1. Setup

Connection settings come from `notebooks/.env` or, per setting, the dbt profile `road_construction_autonomy`
in `~/.dbt/profiles.yml`, the same way as `top_10_cities_trend.ipynb`.

In [ ]:
import json
import os
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import yaml
from dotenv import load_dotenv
from google.cloud import bigquery

# Notebook may be launched from the repo root or from notebooks/
NOTEBOOK_DIR = Path.cwd() if Path.cwd().name == "notebooks" else Path.cwd() / "notebooks"
load_dotenv(NOTEBOOK_DIR / ".env")

In [ ]:
def load_dbt_target(profile="road_construction_autonomy"):
    """Return the active target from the dbt profile, or {} if unavailable."""
    profiles_dir = Path(os.environ.get("DBT_PROFILES_DIR", Path.home() / ".dbt"))
    path = profiles_dir / "profiles.yml"
    if not path.exists():
        return {}
    prof = (yaml.safe_load(path.read_text()) or {}).get(profile, {})
    return prof.get("outputs", {}).get(prof.get("target"), {})


dbt_target = load_dbt_target()

BQ_PROJECT = os.getenv("BQ_PROJECT") or dbt_target.get("project")
BQ_LOCATION = os.getenv("BQ_LOCATION") or dbt_target.get("location", "US")
# dbt's default schema naming: <target dataset>_<custom schema> (see +schema in dbt_project.yml)
BQ_OBT_DATASET = os.getenv("BQ_OBT_DATASET") or (
    f"{dbt_target['dataset']}_obt_road_construction_and_antonomy" if dbt_target.get("dataset") else None
)
BQ_DATASET = os.getenv("BQ_DATASET") or (
    f"{dbt_target['dataset']}_dwh_road_construction" if dbt_target.get("dataset") else None
)

assert BQ_PROJECT and BQ_OBT_DATASET and BQ_DATASET, (
    "Set BQ_PROJECT / BQ_OBT_DATASET / BQ_DATASET in notebooks/.env or configure the dbt profile."
)

SUMMARY_TABLE = f"`{BQ_PROJECT}.{BQ_OBT_DATASET}.route_construction_summary`"
FACT_TABLE = f"`{BQ_PROJECT}.{BQ_DATASET}.fact_construction`"
client = bigquery.Client(project=BQ_PROJECT, location=BQ_LOCATION)

## 2. Load the route summary, construction spots and background constructions

In [ ]:
routes = client.query(f"""
SELECT
  path_name,
  distance_mi,
  construction_count,
  construction_hours,
  high_severity_hours,
  high_severe_impact_hours,
  construction_hours_per_mi,
  ST_ASGEOJSON(path_geography) AS path_geojson,
  ST_ASTEXT(path_geography) AS path_wkt,
  ST_NUMGEOMETRIES(construction_spots) AS distinct_spots
FROM {SUMMARY_TABLE}
""").to_dataframe()
routes = routes.sort_values("construction_hours", ascending=False).reset_index(drop=True)
routes["destination"] = routes["path_name"].str.split(" -> ").str[1]

# One row per construction spot (where a construction meets the route)
spots = client.query(f"""
SELECT path_name, ST_X(spot) AS lon, ST_Y(spot) AS lat
FROM {SUMMARY_TABLE}, UNNEST(ST_DUMP(construction_spots)) AS spot
""").to_dataframe()

# Every 2017-2021 construction inside the routes' bounding box, drawn as background context
background = client.query(f"""
WITH box AS (
  SELECT ST_BOUNDINGBOX(ST_UNION_AGG(path_geography)) AS b FROM {SUMMARY_TABLE}
)
SELECT start_longitude AS lon, start_latitude AS lat
FROM {FACT_TABLE}, box
WHERE start_longitude BETWEEN b.xmin - 0.03 AND b.xmax + 0.03
  AND start_latitude BETWEEN b.ymin - 0.03 AND b.ymax + 0.03
  AND DATE(start_time_local) >= DATE('2017-01-01')
  AND DATE(start_time_local) < DATE('2022-01-01')
""").to_dataframe()

print(f"{len(routes)} routes, {len(spots):,} route construction spots, {len(background):,} background constructions")

## 3. Route table

One row per route, sorted by construction hours. `path_wkt` is the route geometry (paste into BigQuery Geo Viz
or any WKT viewer); the construction spots are in `route_construction_summary.construction_spots`.

In [ ]:
TABLE_COLUMNS = {
    "path_name": "Route",
    "distance_mi": "Distance (mi)",
    "construction_count": "Constructions",
    "construction_hours": "Construction hours",
    "high_severity_hours": "High severity (3–4) hours",
    "high_severe_impact_hours": "High/Severe weather impact hours",
    "construction_hours_per_mi": "Hours per route mile",
    "distinct_spots": "Distinct spots",
}

table = routes[list(TABLE_COLUMNS)].rename(columns=TABLE_COLUMNS)
table["High severity share"] = routes["high_severity_hours"] / routes["construction_hours"]
table["High/Severe impact share"] = routes["high_severe_impact_hours"] / routes["construction_hours"]

(
    table.style
    .format({
        "Distance (mi)": "{:.2f}",
        "Constructions": "{:,}",
        "Construction hours": "{:,.0f}",
        "High severity (3–4) hours": "{:,.0f}",
        "High/Severe weather impact hours": "{:,.1f}",
        "Hours per route mile": "{:,.0f}",
        "Distinct spots": "{:,}",
        "High severity share": "{:.1%}",
        "High/Severe impact share": "{:.2%}",
    })
    .hide(axis="index")
)

## 4. Chart: construction hours on each route

For each route: total construction hours, high severity (3–4) hours, and High/Severe weather-impact hours, side by side.
Weather-impact hours are 10–1,000 times smaller than the others, so their bars are barely visible; the value
labels carry them. Routes are sorted highest construction hours first.

In [ ]:
INK_PRIMARY = "#0b0b0b"
INK_SECONDARY = "#52514e"
GRID = "#e4e3df"
NEUTRAL_BAR = "#8a8984"

# measure -> (legend label, color): first three slots of the validated categorical palette
HOUR_MEASURES = {
    "construction_hours": ("Total construction hours", "#2a78d6"),
    "high_severity_hours": ("High severity (3–4) hours", "#eb6834"),
    "high_severe_impact_hours": ("High/Severe weather impact hours", "#1baf7a"),
}


def compact(x, _pos=None):
    for div, suffix in ((1e6, "M"), (1e3, "k")):
        if abs(x) >= div:
            return f"{x / div:.3g}{suffix}"
    return f"{x:.3g}"


x_labels = routes["path_name"].str.replace(" -> ", " →\n", regex=False)
x = range(len(routes))

fig, ax = plt.subplots(figsize=(12, 6), constrained_layout=True)

bar_w = 0.27
for i, (col, (label, color)) in enumerate(HOUR_MEASURES.items()):
    offsets = [xi + (i - 1) * (bar_w + 0.01) for xi in x]
    bars = ax.bar(offsets, routes[col], width=bar_w, color=color, label=label)
    ax.bar_label(bars, labels=[compact(v) for v in routes[col]], padding=3,
                 fontsize=9, fontweight="bold", color=INK_PRIMARY)

ax.set_xticks(list(x), x_labels, fontsize=10, fontweight="bold", color=INK_PRIMARY)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(compact))
ax.set_ylabel("Construction hours", color=INK_SECONDARY)
ax.set_title("Construction hours on each route, 2017–2021", loc="left",
             fontsize=13, fontweight="bold", color=INK_PRIMARY)
ax.legend(frameon=False, loc="upper right", labelcolor=INK_PRIMARY)
ax.set_ylim(0, routes["construction_hours"].max() * 1.12)
ax.spines[["top", "right"]].set_visible(False)
ax.spines[["left", "bottom"]].set_color(GRID)
ax.grid(axis="y", color=GRID, linewidth=0.8)
ax.set_axisbelow(True)
ax.tick_params(colors=INK_SECONDARY, length=0)

plt.show()

## 5. Map: routes and the constructions on them

Each route is drawn in its own color, with its construction spots on top. A spot is where the construction meets
the route (the middle of the overlapped section), so every spot sits on its route. Grey dots are all other 2017–2021 constructions in the area. Each destination is labeled with its
total construction hours and number of constructions.

In [ ]:
# fixed route -> color, so a route keeps its color however the table is sorted
ROUTE_COLORS = {
    "SF -> San Rafael": "#eb6834",
    "SF -> San Leandro": "#2a78d6",
    "SF -> San Mateo": "#6250d6",
    "SF -> Berkley": "#1baf7a",
    "SF -> Pacifica": "#008300",
}
# destination label offset (points) from the route's end, chosen so labels clear the routes
LABEL_OFFSETS = {
    "SF -> San Rafael": (25, -8),
    "SF -> San Leandro": (-110, -45),
    "SF -> San Mateo": (25, -20),
    "SF -> Berkley": (-10, 25),
    "SF -> Pacifica": (-10, 25),
}

fig, ax = plt.subplots(figsize=(7, 8.5), constrained_layout=True)
ax.set_facecolor("#f6f8fb")
ax.scatter(background["lon"], background["lat"], s=2, color="#c9cdd8", alpha=0.5,
           linewidths=0, label="Other Bay Area constructions", zorder=1)

origin = None
for _, r in routes.iterrows():
    color = ROUTE_COLORS.get(r["path_name"], NEUTRAL_BAR)
    lons, lats = zip(*json.loads(r["path_geojson"])["coordinates"])
    origin = origin or (lons[0], lats[0])
    ax.plot(lons, lats, color=color, linewidth=2.5, solid_capstyle="round", zorder=3,
            label=f"{r['path_name'].replace('->', '→')} ({r['distance_mi']:.2f} mi)")
    on_route = spots[spots["path_name"] == r["path_name"]]
    ax.scatter(on_route["lon"], on_route["lat"], s=22, color=color, alpha=0.35,
               edgecolors="white", linewidths=0.6, zorder=4)
    ax.scatter([lons[-1]], [lats[-1]], s=55, color=color, edgecolors="white", linewidths=1.5, zorder=5)
    ax.annotate(
        f"{r['destination']}\n{r['construction_hours'] / 1e3:,.1f}k hrs · {r['construction_count']:,} constructions",
        xy=(lons[-1], lats[-1]), xytext=LABEL_OFFSETS.get(r["path_name"], (20, 20)), textcoords="offset points",
        fontsize=7, fontweight="bold", color=INK_PRIMARY, zorder=6,
        bbox={"boxstyle": "round,pad=0.3", "fc": "white", "ec": color, "lw": 1.5},
        arrowprops={"arrowstyle": "-|>", "color": color, "lw": 1.2},
    )

ax.scatter(*origin, marker="*", s=220, color="#eda100", edgecolors=INK_PRIMARY, linewidths=1,
           zorder=7, label="Origin: Market & Van Ness (SF)")

# frame the map on the routes, not the whole background box
all_lons = [c[0] for g in routes["path_geojson"] for c in json.loads(g)["coordinates"]]
all_lats = [c[1] for g in routes["path_geojson"] for c in json.loads(g)["coordinates"]]
ax.set_xlim(min(all_lons) - 0.03, max(all_lons) + 0.03)
ax.set_ylim(min(all_lats) - 0.02, max(all_lats) + 0.03)
ax.set_aspect(1 / 0.79)  # 1° lon ≈ 0.79 × 1° lat at ~37.8°N

ax.set_xlabel("Longitude (°W)", fontsize=8, fontweight="bold", color=INK_SECONDARY)
ax.set_ylabel("Latitude (°N)", fontsize=8, fontweight="bold", color=INK_SECONDARY)
ax.set_title("Construction on potential self-driving routes, 2017–2021", loc="left",
             fontsize=11, fontweight="bold", color=INK_PRIMARY)
ax.grid(color="white", linewidth=1)
ax.tick_params(colors=INK_SECONDARY, length=0, labelsize=7)
ax.legend(loc="upper right", fontsize=7, framealpha=0.95, markerscale=1)
plt.show()